# Script to update metadata for any dataset

Check metadata for old sequences to see if anything has been added

Databases: NCBI Virus, Andersen, GISAID

In [2]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 



In [ ]:
# Make sure you have the correct paths

# Dates
start_date = "11-01-2021"
end_date = "07-25-2025"
date_range = start_date + "--" + end_date
update_date = "10-07-2025"

# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"
os.chdir(references)
state_ref = pd.read_csv("states_ref.csv")

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads_gisaid = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder for gisaid
# downloads_gisaid = "C:/Users/maksi/Downloads/"
downloads_gisaid = home + "GISAID/downloads/" + date_range + "_Antarctica_North_America_South_America/" 
downloads_ncbi_virus = home + "NCBI_Virus/downloads/" + date_range + "_Antarctica_North_America_South_America/"

genotype = "A3"
genotype_underscored = genotype.replace(".", "_")

originals = home + "Combinations/Andersen_NCBI_Virus_GISAID/" + date_range + "_Antarctica_North_America_South_America/" # "_cats/" # + "_" + genotype_underscored + "/"
# originals = home + "Trees/" + genotype_underscored + "/" # + date_range + "/" # + genotype_underscored + "/concat/" #trimmed/"

# complete = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "_Antarctica_North_America_South_America/updated_" + update_date + "/"
complete = originals

os.chdir(originals)

## Original Files

In [22]:
# Function to prepare dataframes
def fasta_df_og(file_name, states_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    # segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    identifiers = []
    genotypes = []
    name_states = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    if len(header.split("|")) > 6:
                            identifier = header.split("|")[0]
                            identifiers.append(identifier)
                            split_first_header = split_header[1].split("/")
                    else:
                        identifiers.append("unknown")
                        split_first_header = split_header[0].split("/")
                    # print(split_first_header)
                    # print(split_header)
                    headers.append(header) 
                    name_states.append(split_first_header[2].replace("_", " "))
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[-6]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[-5])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    # segments.append(split_header[].split("_")[-1])
                    host_types.append(split_header[-2])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[-3].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["name_state"] = name_states
    # fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["name_state"].apply(lambda x: 
                                                      
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if states_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if states_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any() 
                                                        # If "x" has neither the state abbreviation nor the full state name
                                                        
                                                        else "USA")
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    if len(identifiers) == len(fasta):
        fasta["Identifier"] = identifiers
    
    return fasta

original_fasta_dfs = {}

for dirpath, dirs, files in os.walk(originals):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fas" in file_name and genotype in file_name: 
            fasta_file = fasta_df_og(file_name, state_ref)
            original_fasta_dfs[file_name] = fasta_file
            # print(fasta_file)
            # break 
    break 

In [19]:
print(list(original_fasta_dfs.keys())[0])
print(original_fasta_dfs["C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_D1.3_HA_combined_11-01-2021--07-25-2025.fasta"])

C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_D1.3_HA_combined_11-01-2021--07-25-2025.fasta
                                                Header     Isolate_Id  \
0    SRR32254505|A/turkey/United_States/25-002414-0...  25-002414-001   
1    SRR32254524|A/chicken/United_States/25-002299-...  25-002299-002   
2    SRR32254525|A/chicken/United_States/25-002299-...  25-002299-001   
3    SRR32254527|A/turkey/United_States/25-002298-0...  25-002298-002   
4    SRR32254528|A/turkey/United_States/25-002298-0...  25-002298-001   
..                                                 ...            ...   
342  EPI_ISL_19777110|A/chicken/Ohio/003106-001/202...     003106-001   
343  EPI_ISL_19777105|A/turkey/Ohio/003240-001/2025...     003240-001   
344  EPI_ISL_19777107|A/turkey/Ohio/003238-001/2025...     003238-001   
345  EPI_ISL_19777106|A/chicken/Ohio/003239-001/202...   

## GISAID

In [ ]:
# # Get data from GISAID

# username = input("Username: ")
# password = input("Password: ")
# browser = input("Browser: ")
# sleep_time = input("Seconds to sleep in between clicks: ")
# continent = input("Continent(s) separated by commas: ")
# start_date = dateutil.parser.parse(start_date).strftime("%Y-%m-%d") # Make sure date is in correct format
# end_date = dateutil.parser.parse(end_date).strftime("%Y-%m-%d")

# open_gisaid(username, password, browser, sleep_time, continent, start_date, end_date)

In [ ]:
# Get downloaded GISAID data

all_metadata_files = []
all_fasta_files = []

# Grab files
for dirpath, dirs, files in os.walk(downloads_gisaid):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, state_ref) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)

In [ ]:
# Search for dates and states based on isolate

genotype_keys = {}
for og_key in original_fasta_dfs:
    key = "_".join(og_key.split("/")[-1].split("_")[0:2])
    print(key)
    print(og_key)
    if key not in genotype_keys.keys(): # If we haven't seen this genotype before
        og_df = original_fasta_dfs[og_key]
        # print(og_df)
        og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
        og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else 0)
        og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
        # og_df["Identifier"] = ""
        og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
        for isolate in og_df_update_needed["Isolate_Id"].values:
            # print(isolate)
            for downloaded_df in all_fasta_files:
                if isolate in downloaded_df["Isolate_Id"].values:
                    # print(isolate)
                    og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = downloaded_df.loc[downloaded_df[downloaded_df["Isolate_Id"] == isolate].index[0], "Geo_Location"]
                    og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = downloaded_df.loc[downloaded_df[downloaded_df["Isolate_Id"] == isolate].index[0], "Collection_Date"]
                    og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = downloaded_df.loc[downloaded_df[downloaded_df["Isolate_Id"] == isolate].index[0], "Identifier"] # .split("|")[0]

                    # Truncate date to year if autocompleted to 1/1
                    # og_df_update_needed["Date Collected"] = og_df_update_needed["Date Collected"].apply(lambda x: print(str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year)) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x)

        og_df_update_needed = og_df_update_needed.drop_duplicates(subset="Header", keep="first")
        print(og_df_update_needed)

        # new_df = og_df.merge(og_df_update_needed, how="left")
        new_df = og_df.set_index('Header')
        new_df.update(og_df_update_needed.set_index('Header'))
        new_df = new_df.reset_index()
        # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates(['Isolate_Id'], keep="last")
        new_df["Header"] = new_df["Identifier"] + "|" + new_df["Isolate_Name"] + "|H5N1|" + new_df["Geo_Location"] + "|" + new_df["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + new_df["Host_Type"] + "|" + new_df["Genotype"]
        print(new_df)
        # break 

        original_fasta_dfs[og_key] = new_df
        genotype_keys[key]= new_df["Header"]
    else:
        original_fasta_dfs[og_key]["Header"] = genotype_keys[key] # Substitute with new dataset

# b3_13_df = original_fasta_dfs["C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Alignments/11-01-2021--07-04-2025/D1_3_11-01-2021--06-27-2025_concat_315.fas"]
# print(b3_13_df[b3_13_df["Date Collected"] == "2025-01-01"])

D1_3
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Trees/D1_3/D1_3_11-01-2021--07-04-2025_concat_318.fas
                                                Header     Isolate_Id  \
0    SRR32254505|A/turkey/USA/25-002414-001/2025|H5...  25-002414-001   
1    SRR32254524|A/chicken/USA/25-002299-002/2025|H...  25-002299-002   
2    SRR32254525|A/chicken/USA/25-002299-001/2025|H...  25-002299-001   
3    SRR32254527|A/turkey/USA/25-002298-002/2025|H5...  25-002298-002   
4    SRR32254528|A/turkey/USA/25-002298-001/2025|H5...  25-002298-001   
..                                                 ...            ...   
313  SRR33682320|A/red-tailed_hawk/USA/25-013658-00...  25-013658-008   
314  SRR33682323|A/osprey/USA/25-014057-001/2025|H5...  25-014057-001   
315  SRR33682327|A/hawk/USA/25-011271-004/2025|H5N1...  25-011271-004   
316  SRR33764585|A/hawk/USA/25-011271-004/2025|H5N1...  25-011271-004   
317  SRR33764497|A/turkey/USA/25-005337-001/2025|H5...  25-005337-001   

          

## Andersen

In [ ]:
# Read metadata

metadata_folder = home + "Andersen/avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t") # Everything else
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")

print(len(metadata))

print(metadata.columns)

# Merge with genbank_mapping
genbank_mapping = genbank_mapping.rename(columns={"sra_run":"Run"})
metadata = metadata.merge(genbank_mapping, how="inner")
metadata["name_state"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# Get geolocation for second state attribute

os.chdir(references)
state_ref = pd.read_csv("states_ref.csv")
# Format: USA-[state abbreviation], e.g. USA-MD
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(", ", " ").split(" "))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(', ', ' ').split(' ')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(", ", " ").split(" "))), regex=True).any() 
                                                        # If "x" has neither the state abbreviation nor the full state name
                                                        else 
                                                        x)

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) 
display(metadata)

10228
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
45776


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,is_retracted,retraction_detection_date_utc,seg_file,seg_seq_name,seg,genbank_acc,genbank_seg,genbank_name,name_state,Geo_Location
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,False,NaN,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
1,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,False,NaN,SRR28752446_MP_cns.fa,Consensus_SRR28752446_MP_cns_threshold_0.5_qua...,MP,PP740723.1,7,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
2,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,False,NaN,SRR28752446_NA_cns.fa,Consensus_SRR28752446_NA_cns_threshold_0.5_qua...,NaN,PP740724.1,6,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
3,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,False,NaN,SRR28752446_NP_cns.fa,Consensus_SRR28752446_NP_cns_threshold_0.5_qua...,NP,PP740725.1,5,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
4,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,False,NaN,SRR28752446_NS_cns.fa,Consensus_SRR28752446_NS_cns_threshold_0.5_qua...,NS,PP740726.1,8,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45771,SRR34270144,WGS,147.58,379458191,PRJNA980729,SAMN49684765,Viral,133079770,USDA-NVSL,2025-03-23,...,False,NaN,SRR34270144_NP_cns.fa,Consensus_SRR34270144_NP_cns_threshold_0.5_qua...,NP,PV949088.1,5,A/Duck/MD/25-010529-001-original/2025,MD,USA-MD
45772,SRR34270144,WGS,147.58,379458191,PRJNA980729,SAMN49684765,Viral,133079770,USDA-NVSL,2025-03-23,...,False,NaN,SRR34270144_NS_cns.fa,Consensus_SRR34270144_NS_cns_threshold_0.5_qua...,NS,PV949091.1,8,A/Duck/MD/25-010529-001-original/2025,MD,USA-MD
45773,SRR34270144,WGS,147.58,379458191,PRJNA980729,SAMN49684765,Viral,133079770,USDA-NVSL,2025-03-23,...,False,NaN,SRR34270144_PA_cns.fa,Consensus_SRR34270144_PA_cns_threshold_0.5_qua...,PA,PV949086.1,3,A/Duck/MD/25-010529-001-original/2025,MD,USA-MD
45774,SRR34270144,WGS,147.58,379458191,PRJNA980729,SAMN49684765,Viral,133079770,USDA-NVSL,2025-03-23,...,False,NaN,SRR34270144_PB1_cns.fa,Consensus_SRR34270144_PB1_cns_threshold_0.5_qu...,PB1,PV949085.1,2,A/Duck/MD/25-010529-001-original/2025,MD,USA-MD


In [ ]:
# Collapse dataset to only sequences in original dataset AND without dates OR states

genotype_keys = {}

for og_key in original_fasta_dfs:
    print(og_key)
    key = "_".join(og_key.split("/")[-1].split("_")[0:2])
    print(key)
    if key not in genotype_keys.keys():
        og_df = original_fasta_dfs[og_key]
        print(og_df)
        # print(og_df)
        og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
        og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).month == dateutil.parser.parse("2025-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).day == dateutil.parser.parse("2025-01-01").day else 0)
        og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
        # og_df["Identifier"] = ""
        og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
        for isolate in og_df_update_needed["Isolate_Id"].values:
            # print(isolate)
            # for new_df in metadata:
            if isolate in metadata["isolate"].values:
                # print(isolate)
                # print(metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "geo_loc_name"])
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "Geo_Location"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "Collection_Date"]
                # Update the genbank name too
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Isolate_Name"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "genbank_name"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "Run"]

        print(og_df_update_needed)

        # new_df = og_df.merge(og_df_update_needed, how="left")
        # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")
        new_df = og_df.set_index('Header')
        new_df.update(og_df_update_needed.set_index('Header'))
        new_df = new_df.reset_index()
        new_df["Header"] = new_df["Identifier"] + "|" + new_df["Isolate_Name"] + "|" + "H5N1" + "|" + new_df["Geo_Location"] + "|" + new_df["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else dateutil.parser.parse(str(x)).strftime("%Y-%m-%d")) + "|" + new_df["Host_Type"] + "|" + new_df["Genotype"]

        # print(new_df)
        # break 

        original_fasta_dfs[og_key] = new_df
        genotype_keys[key]= new_df["Header"]
    else:
        original_fasta_dfs[og_key]["Header"] = genotype_keys[key]
    # break # Only do this once, then copy metadata over later
    
# b3_13_df = original_fasta_dfs["C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Alignments/11-01-2021--07-04-2025/B3_13_HA_11-01-2021--07-04-2025_aln.fasta"]
# print(b3_13_df[b3_13_df["Header"].str.contains("2025-01-02")])

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Trees/D1_3/D1_3_11-01-2021--07-04-2025_concat_318.fas
D1_3
                                                Header     Isolate_Id  \
0    SRR32254505|A/turkey/USA/25-002414-001/2025|H5...  25-002414-001   
1    SRR32254524|A/chicken/USA/25-002299-002/2025|H...  25-002299-002   
2    SRR32254525|A/chicken/USA/25-002299-001/2025|H...  25-002299-001   
3    SRR32254527|A/turkey/USA/25-002298-002/2025|H5...  25-002298-002   
4    SRR32254528|A/turkey/USA/25-002298-001/2025|H5...  25-002298-001   
..                                                 ...            ...   
313  SRR33682320|A/red-tailed_hawk/USA/25-013658-00...  25-013658-008   
314  SRR33682323|A/osprey/USA/25-014057-001/2025|H5...  25-014057-001   
315  SRR33682327|A/hawk/USA/25-011271-004/2025|H5N1...  25-011271-004   
316  SRR33764585|A/hawk/USA/25-011271-004/2025|H5N1...  25-011271-004   
317  SRR33764497|A/turkey/USA/25-005337-001/2025|H5...  25-005337-001   

          

In [ ]:
print(dateutil.parser.parse("02-Jan-2025").strftime("%Y-%m-%d"))

2025-01-02


## NCBI Virus

In [ ]:

downloads_ncbi_virus = home + "NCBI_Virus/downloads/" + date_range + "_Antarctica_North_America_South_America/"

os.chdir(downloads_ncbi_virus)

# Read metadata
ncbi_metadata = pd.read_csv("sequences.csv")

# Integrate genotypes
# os.chdir(downloads_ncbi_virus + "11-01-2021--04-14-2025/")
# output = pd.read_csv("output.tsv", delimiter="\t")

# os.chdir(downloads_ncbi_virus + "04-14-2025--05-14-2025/")
# may_output = pd.read_csv("output.tsv", delimiter="\t")

print(ncbi_metadata)

        Accession GenBank_RefSeq SRA_Accession BioSample BioProject  \
0      PP907915.1        GenBank           NaN       NaN        NaN   
1      PP907916.1        GenBank           NaN       NaN        NaN   
2      PP907917.1        GenBank           NaN       NaN        NaN   
3      PP907918.1        GenBank           NaN       NaN        NaN   
4      PP907919.1        GenBank           NaN       NaN        NaN   
...           ...            ...           ...       ...        ...   
86114  OK205883.1        GenBank           NaN       NaN        NaN   
86115  OK205884.1        GenBank           NaN       NaN        NaN   
86116  OK205885.1        GenBank           NaN       NaN        NaN   
86117  OK205886.1        GenBank           NaN       NaN        NaN   
86118  OK205887.1        GenBank           NaN       NaN        NaN   

           Organism_Name                         Species Genotype    Isolate  \
0      Influenza A virus  Alphainfluenzavirus influenzae     H5N1  

In [ ]:
# Search for dates and states based on isolate

genotype_keys = {}

for og_key in original_fasta_dfs:
    key = "_".join(og_key.split("/")[-1].split("_")[0:2])
    print(key)
    print(og_key)
    if key not in genotype_keys.keys():
        og_df = original_fasta_dfs[og_key]
        print(og_df)
        # print(og_df)
        og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
        og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).month == dateutil.parser.parse("2025-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).day == dateutil.parser.parse("2025-01-01").day else 0)

        # og_df["Unknown_Dates"] = og_df["Date Collected"].apply(dateutil.parser.parse).apply(lambda x: 1 if x == dateutil.parser.parse("2023-01-01") or x == dateutil.parser.parse("2024-01-01") or x == dateutil.parser.parse("2025-01-01") else 0)
        og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
        # og_df["Identifier"] = ""
        og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
        # print(og_df_update_needed)
        for isolate in og_df_update_needed["Isolate_Id"].values:
            # print(isolate)
            # for new_df in ncbi_metadata:
                # print(new_df)
            if isolate in ncbi_metadata["Isolate"].values:
                print(isolate)
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Geo_Location"] # .replace(": ", "-")
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Collection_Date"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "SRA_Accession"]

        # new_df = og_df.merge(og_df_update_needed, how="left")
        # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")

        new_df = og_df.set_index('Header')
        new_df.update(og_df_update_needed.set_index('Header'))
        new_df = new_df.reset_index()
        new_df["Header"] = new_df["Identifier"].apply(lambda x: x.split(",")[0]) + "|" + new_df["Isolate_Name"] + "|H5N1|" + new_df["Geo_Location"] + "|" + new_df["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else dateutil.parser.parse(str(x)).strftime("%Y-%m-%d")) + "|" + new_df["Host_Type"] + "|" + new_df["Genotype"]

        print(new_df)
        # break 

        original_fasta_dfs[og_key] = new_df
        genotype_keys[key]= new_df["Header"]
    else:
        original_fasta_dfs[og_key]["Header"] = genotype_keys[key]
    # break 
    

D1_3
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Trees/D1_3/D1_3_11-01-2021--07-04-2025_concat_318.fas
                                                Header     Isolate_Id  \
0    SRR32254505|A/turkey/USA/25-002414-001/2025|H5...  25-002414-001   
1    SRR32254524|A/chicken/USA/25-002299-002/2025|H...  25-002299-002   
2    SRR32254525|A/chicken/USA/25-002299-001/2025|H...  25-002299-001   
3    SRR32254527|A/turkey/USA/25-002298-002/2025|H5...  25-002298-002   
4    SRR32254528|A/turkey/USA/25-002298-001/2025|H5...  25-002298-001   
..                                                 ...            ...   
313  SRR33682320|A/red-tailed_hawk/USA/25-013658-00...  25-013658-008   
314  SRR33682323|A/osprey/USA/25-014057-001/2025|H5...  25-014057-001   
315  SRR33682327|A/hawk/USA/25-011271-004/2025|H5N1...  25-011271-004   
316  SRR33764585|A/hawk/USA/25-011271-004/2025|H5N1...  25-011271-004   
317  SRR33764497|A/Turkey/OH/25-005337-001-original...  25-005337-001   

          

In [ ]:
# print(list(original_fasta_dfs.keys())[0])
# print(original_fasta_dfs["C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Alignments/11-01-2021--07-04-2025/D1_3_11-01-2021--06-27-2025_concat_315.fas"])

## De-duplicate if needed

In [27]:
print(original_fasta_dfs["C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Combinations/Andersen_NCBI_Virus_GISAID/11-01-2021--07-25-2025_Antarctica_North_America_South_America/all_D1.3_NS_combined_11-01-2021--07-25-2025.fasta"])
print(len(original_fasta_dfs))

# De-duplicate based on partial isolates and years
def partial_isolate(id):

    partial = id.split("_")[-1] # If 25_, get the last bit
    digits = partial.split("-")
    # Build partial isolates
    isolate = ""
    other = ""
    for d in digits:
        # print(d)
        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
            isolate = d + "-"
        elif len(d) == 3 and d.isnumeric():
            isolate = isolate + d
        elif d.isnumeric() == False: # If it's a weird isolate
            other = d + "-"
        else: 
            other = other + d
    # Now add to list to check in Andersen files without doing wild for loops
    if len(isolate) == 10: # If this is a correctly formatted isolate
        # isolates.append(isolate)
        # All headers are followed by sequences
        partial_isolate = isolate
    else: # If this is some other isolate
        partial_isolate = other

    return partial_isolate

for og_key in original_fasta_dfs:
    df = original_fasta_dfs[og_key]
    df["Partials"] = df["Isolate_Id"].apply(partial_isolate)
    df["Year"] = df["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
    df = df.drop_duplicates(subset=["Partials", "Year"], keep="first")
    print(df)

                                                Header     Isolate_Id  \
0    SRR32254505|A/turkey/United_States/25-002414-0...  25-002414-001   
1    SRR32254524|A/chicken/United_States/25-002299-...  25-002299-002   
2    SRR32254525|A/chicken/United_States/25-002299-...  25-002299-001   
3    SRR32254527|A/turkey/United_States/25-002298-0...  25-002298-002   
4    SRR32254528|A/turkey/United_States/25-002298-0...  25-002298-001   
..                                                 ...            ...   
342  EPI_ISL_19777110|A/chicken/Ohio/003106-001/202...     003106-001   
343  EPI_ISL_19777105|A/turkey/Ohio/003240-001/2025...     003240-001   
344  EPI_ISL_19777107|A/turkey/Ohio/003238-001/2025...     003238-001   
345  EPI_ISL_19777106|A/chicken/Ohio/003239-001/202...     003239-001   
346  EPI_ISL_19744893|A/chicken/Puerto_Rico/25-0004...  25-000491-003   

                                   Isolate_Name Subtype     name_state  \
0     A/turkey/United_States/25-002414-001/2025  

# Put it all together

In [20]:
# Copy metadata to all segments

# Get the metadata for the first one
metadata_df = pd.DataFrame()
for og_key in original_fasta_dfs:
    # Use identifier to merge datasets
    metadata_df = original_fasta_dfs[og_key]
    break 

new_fasta_dfs = {}
for og_key in original_fasta_dfs: # For the rest, including the first
    # Do NOT overwrite sequence
    df = original_fasta_dfs[og_key]
    sequences = df["Sequence"]
    # print(sequences)
    new_df = metadata_df.combine_first(df) # Overwrite everything in the second
    new_df["Sequence"] = sequences # Except the sequences, put those back
    new_fasta_dfs[og_key] = new_df

In [ ]:
def df_to_fasta(fasta, file_name, output_path):

    output_file = open(output_path + file_name, "w")

    for index, row in fasta.iterrows():
        name = fasta.loc[index, "full_header"]
        print(name)
        sequence = fasta.loc[index, "sequence"]
    # First is header, second is sequence
        output_file.write(str(name) + "\n")
        output_file.write(str(sequence) + "\n")
    output_file.close()

for key in new_fasta_dfs:
    df = new_fasta_dfs[key]
    # df["Identifier"] = df["Identifier"].apply(lambda x: "unknown" if x != x else x) # Put "unknown" if NaN
    # df = df[df["Genotype"] == genotype] # Make sure we only have the genotype we want
    df["Header"] = df["Header"].fillna("unknown")
    df["full_header"] = ">" + df["Header"]
    df["sequence"] = df["Sequence"]

    df_to_fasta(df, key.split("/")[-1][:-6] + "_updated_" + update_date + ".fasta", complete)

>SRR32254505|A/turkey/USA/25-002414-001/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254524|A/chicken/USA/25-002299-002/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254525|A/chicken/USA/25-002299-001/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254527|A/turkey/USA/25-002298-002/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254528|A/turkey/USA/25-002298-001/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254529|A/turkey/USA/25-002297-004/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254530|A/turkey/USA/25-002297-003/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254531|A/turkey/USA/25-002297-002/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254532|A/turkey/USA/25-002297-001/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254533|A/turkey/USA/25-002295-004/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254534|A/turkey/USA/25-002295-003/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254535|A/turkey/USA/25-002295-002/2025|H5N1|USA|2025|domestic_avian|D1.3
>SRR32254545|A/chicken/USA/25-002277-002/2025|H5N1